In [1]:
# Force vLLM to use spawn method to avoid CUDA fork errors in Jupyter
import os
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

# Tell the C++ compiler's linker exactly where to find Conda's CUDA libraries
conda_prefix = os.environ.get("CONDA_PREFIX", "/home/dylan/miniconda3")
os.environ["LIBRARY_PATH"] = f"{conda_prefix}/lib:" + os.environ.get("LIBRARY_PATH", "")

In [2]:
import torch
import time
from data import *
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

In [3]:
# Set seeds
seed = 43
np.random.seed(seed);
torch.manual_seed(seed);

In [4]:
# Configuration
# For full dataset: n_samples = 15000, max_new_tokens = 100, batch_size = 16
model = "Qwen/Qwen1.5-MoE-A2.7B-Chat-GPTQ-Int4"
n_samples = 1000
max_new_tokens = 100
sampling_params = SamplingParams(max_tokens=max_new_tokens)
batch_size = 16
max_context_len = 4096
n_warmup_samples = 2

# Load model and tokenizer
print(f"\nInitializing vLLM...")
llm = LLM(model=model, trust_remote_code=True, max_model_len=max_context_len, max_num_seqs=batch_size)
tokenizer = AutoTokenizer.from_pretrained(model, trust_remote_code=True)


Initializing vLLM...
INFO 04-22 14:25:10 [utils.py:233] non-default args: {'trust_remote_code': True, 'max_model_len': 4096, 'max_num_seqs': 16, 'disable_log_stats': True, 'model': 'Qwen/Qwen1.5-MoE-A2.7B-Chat-GPTQ-Int4'}


INFO 04-22 14:25:11 [model.py:549] Resolved architecture: Qwen2MoeForCausalLM
INFO 04-22 14:25:11 [model.py:1678] Using max model len 4096
INFO 04-22 14:25:12 [gptq_marlin.py:229] The model is convertible to gptq_marlin during runtime. Using gptq_marlin kernel.
INFO 04-22 14:25:12 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-22 14:25:12 [vllm.py:790] Asynchronous scheduling is enabled.
(EngineCore pid=834093) INFO 04-22 14:25:21 [core.py:105] Initializing a V1 LLM engine (v0.19.0) with config: model='Qwen/Qwen1.5-MoE-A2.7B-Chat-GPTQ-Int4', speculative_config=None, tokenizer='Qwen/Qwen1.5-MoE-A2.7B-Chat-GPTQ-Int4', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=4096, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_

(EngineCore pid=834093) <frozen importlib._bootstrap_external>:1184: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(EngineCore pid=834093) <frozen importlib._bootstrap_external>:1184: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.
(EngineCore pid=834093) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  33% Completed | 1/3 [00:02<00:04,  2.03s/it]
Loading safetensors checkpoint shards:  67% Completed | 2/3 [00:04<00:02,  2.15s/it]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:04<00:00,  1.26s/it]
Loading safetensors checkpoint shards: 100% Completed | 3/3

(EngineCore pid=834093) INFO 04-22 14:25:28 [default_loader.py:384] Loading weights took 4.47 seconds
(EngineCore pid=834093) INFO 04-22 14:25:29 [gpu_model_runner.py:4820] Model loading took 7.83 GiB memory and 6.597074 seconds
(EngineCore pid=834093) INFO 04-22 14:25:33 [backends.py:1051] Using cache directory: /home/dylan/.cache/vllm/torch_compile_cache/d0279df09c/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=834093) INFO 04-22 14:25:33 [backends.py:1111] Dynamo bytecode transform time: 3.46 s
(EngineCore pid=834093) INFO 04-22 14:25:35 [backends.py:285] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 1.207 s
(EngineCore pid=834093) INFO 04-22 14:25:35 [decorators.py:303] Directly load AOT compilation from path /home/dylan/.cache/vllm/torch_compile_cache/torch_aot_compile/6a0121729d110e99793c55be7a0916976ef61a82b846da14503d0e81c2fa339b/rank_0_0/model
(EngineCore pid=834093) INFO 04-22 14:25:35 [monitor.py:48] torch.compile took 5.32 

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 7/7 [00:00<00:00, 10.80it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 5/5 [00:00<00:00, 12.40it/s]


(EngineCore pid=834093) INFO 04-22 14:25:40 [gpu_model_runner.py:6046] Graph capturing finished in 2 secs, took 0.17 GiB
(EngineCore pid=834093) INFO 04-22 14:25:40 [gpu_worker.py:597] CUDA graph pool memory: 0.17 GiB (actual), 0.54 GiB (estimated), difference: 0.37 GiB (222.1%).
(EngineCore pid=834093) INFO 04-22 14:25:40 [core.py:283] init engine (profile, create kv cache, warmup model) took 10.94 seconds
(EngineCore pid=834093) INFO 04-22 14:25:42 [vllm.py:790] Asynchronous scheduling is enabled.


In [5]:
# Load MMLU dataset
print("Loading MMLU dataset...")
dataset = get_data_mmlu(n_samples=n_samples, shuffle_seed=seed)

# Set chat template
prompts = []
for d in dataset:
    messages = [
        {
            "role": "system", 
            "content": "You are a logical reasoning assistant. You must provide all of your reasoning, explanations, and final answers entirely in English. Do not use any other language."
        },
        {
            "role": "user", 
            "content": (
                f"The following is a multiple-choice question.\n"
                f"Question: {d['question']}\n"
                f"A) {d['choices'][0]}\nB) {d['choices'][1]}\nC) {d['choices'][2]}\nD) {d['choices'][3]}\n\n"
                f"Do not simply output the letter. Think step-by-step, carefully explaining your "
                f"reasoning for each option before arriving at the final answer. Your entire response must be strictly in English."
            )
        }
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    prompts.append(prompt)

Loading MMLU dataset...
Streaming cais/mmlu (all) (samples: 1000)...


In [6]:
# Warmup
print("Running warmup...")
llm.generate(prompts[:n_warmup_samples], sampling_params, use_tqdm=False)

Running warmup...


[RequestOutput(request_id=0, prompt='<|im_start|>system\nYou are a logical reasoning assistant. You must provide all of your reasoning, explanations, and final answers entirely in English. Do not use any other language.<|im_end|>\n<|im_start|>user\nThe following is a multiple-choice question.\nQuestion: Which of the following is (are) characteristic of mass spectrometry?\nI. Analyte molecules are converted to gaseous ions.\nII. The ions are separated according to their mass-to-charge ratio.\nIII. In addition to compound identification, mass spectra can be utilized to determine precise isotopic masses and isotopic ratios.\nA) II only\nB) I and II only\nC) I and III only\nD) I, II, and III\n\nDo not simply output the letter. Think step-by-step, carefully explaining your reasoning for each option before arriving at the final answer. Your entire response must be strictly in English.<|im_end|>\n<|im_start|>assistant\n', prompt_token_ids=[151644, 8948, 198, 2610, 525, 264, 19819, 32711, 1784

In [7]:
# Benchmark
print("Running benchmark...")

# Start timing
start_time = time.perf_counter()

outputs = llm.generate(prompts, sampling_params, use_tqdm=True)

# Stop timing
end_time = time.perf_counter()

Running benchmark...


Rendering prompts:   0%|          | 0/1000 [00:00<?, ?it/s]

Processed prompts:   0%|                      | 0/1000 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output:…

In [8]:
# Calculate TPOT over entire dataset
# For now, assume prefill time << decode time
total_time = end_time - start_time
total_output_tokens = sum(len(output.outputs[0].token_ids) for output in outputs)
throughput = total_output_tokens / total_time # Tokens / s
avg_tpot = (total_time / total_output_tokens) * 1000 # ms / token

print("\n" + "=" * 50)
print(f"RESULTS:")
print(f"Total Wall-clock Time: {total_time:.2f} seconds")
print(f"Total Output Tokens:   {total_output_tokens}")
print(f"Throughput:            {throughput:.2f} tokens/sec")
print(f"Average TPOT:          {avg_tpot:.2f} ms/token")
print("=" * 50 + "\n")


RESULTS:
Total Wall-clock Time: 136.79 seconds
Total Output Tokens:   99778
Throughput:            729.41 tokens/sec
Average TPOT:          1.37 ms/token



In [10]:
prompt_idx = 0
print(outputs[prompt_idx].prompt)
outputs[prompt_idx].outputs[0].text

<|im_start|>system
You are a logical reasoning assistant. You must provide all of your reasoning, explanations, and final answers entirely in English. Do not use any other language.<|im_end|>
<|im_start|>user
The following is a multiple-choice question.
Question: Which of the following is (are) characteristic of mass spectrometry?
I. Analyte molecules are converted to gaseous ions.
II. The ions are separated according to their mass-to-charge ratio.
III. In addition to compound identification, mass spectra can be utilized to determine precise isotopic masses and isotopic ratios.
A) II only
B) I and II only
C) I and III only
D) I, II, and III

Do not simply output the letter. Think step-by-step, carefully explaining your reasoning for each option before arriving at the final answer. Your entire response must be strictly in English.<|im_end|>
<|im_start|>assistant



'Step 1: Analyze the options.\nWe need to determine which of the properties are specific to mass spectrometry.\n\nOption I: Analyte molecules are converted to gaseous ions. This is a characteristic of ionization techniques in mass spectrometry, such as chemical ionization and electron ionization.\n\nOption II: The ions are separated according to their mass-to-charge ratio. Mass spectrometry separates ions based on their mass, not necessarily their charge. However, this is a'